In [1]:
print("hi")

hi


In [2]:
links={
    "Coding":[
        "https://www.youtube.com/watch?v=eJtD5vl00HA",
        "https://www.youtube.com/watch?v=6eBSHbLKuN0",
        "https://www.youtube.com/watch?v=p8Za5MtyVdg",
        "https://www.youtube.com/watch?v=8AWEPx5cHWQ"
    ],
    "News":[
        "https://www.youtube.com/watch?v=qKSXMDXOMek",
        "https://www.youtube.com/watch?v=65dNUm2f6NI",
        "https://www.youtube.com/watch?v=oHkE-B5Xhvc"
    ],
    "Podcast":[
        "https://www.youtube.com/watch?v=SfC-cnSgO1E",
        "https://www.youtube.com/watch?v=GYt5093aCQM&pp=0gcJCfwJAYcqIYzv",
        "https://www.youtube.com/watch?v=Pmd6knanPKw"
    ],
    "Interview":[
        "https://youtu.be/uCiBkaA4V6M?si=Y0bo4TZjYSBopyEA",
        "https://youtu.be/1qw5ITr3k9E?si=NUNHf42afFFZZwRz",
        "https://youtu.be/ZjmzEUY1x3Y?si=KacY9nljJS_yUQO6"
    ],
    "Comedy":[
        "https://youtu.be/1t1_a1BZ04o?si=A7YWPLq-wj1vG6rk",
        "https://youtu.be/314OLE6mKOo?si=i8paQVQrqH8V2Rj0",
        "https://youtu.be/W8-n2o8CaFU?si=eHANe5ADjlu3BXFO"
    ],
    "reality show":[
        "https://youtu.be/EU96s14zgLk?si=0a-frvrhbLpnurIz",
        "https://youtu.be/Do0kQEadPpA?si=FZkc8rbs-59fH_7L",
        "https://youtu.be/-Nwa2U-7sQA?si=qgi6dSe10vwbyPyP"
    ],
    "tutorials":[
        "https://www.youtube.com/watch?v=b-UZJVdLbXc",
        "https://www.youtube.com/watch?v=8wUUMOKAK-c&t=53s",
        "https://www.youtube.com/watch?v=wXvljefXyEo&t=3s"
    ],
    "cooking":[
        "https://www.youtube.com/watch?v=QL2tNKHB6ew",
        "https://www.youtube.com/watch?v=P6W8kwmwcno",
        "https://www.youtube.com/watch?v=i_164rISukM"
    ]
}

In [9]:
links={

    "reality show":[
        "https://youtu.be/EU96s14zgLk?si=0a-frvrhbLpnurIz",
        "https://youtu.be/Do0kQEadPpA?si=FZkc8rbs-59fH_7L",
        "https://youtu.be/-Nwa2U-7sQA?si=qgi6dSe10vwbyPyP"
    ],

    "cooking":[
        "https://www.youtube.com/watch?v=QL2tNKHB6ew",
        "https://www.youtube.com/watch?v=P6W8kwmwcno",
        "https://www.youtube.com/watch?v=i_164rISukM"
    ]
}

In [3]:
from yt_rag.helper.get_id_from_youtube_url import get_video_id

In [10]:
categories={}
for item,value in links.items():
    vid_id=[]
    for link in value:
        vid_id.append(get_video_id(link))
        categories[item]=vid_id

In [11]:
import dagshub
dagshub.init(repo_owner='Prahaladha-Reddy', repo_name='YT_RAG', mlflow=True)

Initialized MLflow to track repo "Prahaladha-Reddy/YT_RAG"

Repository Prahaladha-Reddy/YT_RAG initialized!

In [12]:
import mlflow
import numpy as np
import os
from evals.match_queries import evaluate_retrieval 

In [13]:
mlflow.set_experiment("test_retrieval_version0")

k_values = [1, 3, 5]
modalities = ["text", "image"]

In [14]:
for category, video_ids in categories.items():
    category_aggregates = {mod: {"mrr": [], "mean_rank": [], "avg_sim": [], "hit_rates": {k: [] for k in k_values}} for mod in modalities}
    
    for video_id in video_ids:
        for modality in modalities:
            metrics = evaluate_retrieval(video_id, modality=modality, k_values=k_values)
            if not metrics:
                print(f"Skipping {video_id} ({modality}): No metrics returned")
                continue
            
            with mlflow.start_run(run_name=f"{category}_{video_id}_{modality}"):
                mlflow.log_param("category", category)
                mlflow.log_param("video_id", video_id)
                mlflow.log_param("modality", modality)
                mlflow.log_param("total_queries", metrics.get("total_queries", 0))
                mlflow.log_metric("mrr", metrics["mrr"])
                mlflow.log_metric("mean_rank", metrics["mean_rank"])
                mlflow.log_metric("avg_similarity", metrics["avg_similarity"])
                for k, hr in metrics["hit_rates"].items():
                    mlflow.log_metric(f"hit_rate{k}", hr)
            
            category_aggregates[modality]["mrr"].append(metrics["mrr"])
            category_aggregates[modality]["mean_rank"].append(metrics["mean_rank"])
            category_aggregates[modality]["avg_sim"].append(metrics["avg_similarity"])
            for k in k_values:
                category_aggregates[modality]["hit_rates"][k].append(metrics["hit_rates"][k])
    
    with mlflow.start_run(run_name=f"{category}_aggregate"):
        mlflow.log_param("category", category)
        for modality in modalities:
            agg = category_aggregates[modality]
            if agg["mrr"]:
                mlflow.log_metric(f"{modality}_avg_mrr", np.mean(agg["mrr"]))
                mlflow.log_metric(f"{modality}_avg_mean_rank", np.mean(agg["mean_rank"]))
                mlflow.log_metric(f"{modality}_avg_similarity", np.mean(agg["avg_sim"]))
                for k in k_values:
                    mlflow.log_metric(f"{modality}_avg_hit_rate{k}", np.mean(agg["hit_rates"][k]))

No data for video_id EU96s14zgLk in text table.
Skipping EU96s14zgLk (text): No metrics returned
No data for video_id EU96s14zgLk in image table.
Skipping EU96s14zgLk (image): No metrics returned
No data for video_id Do0kQEadPpA in text table.
Skipping Do0kQEadPpA (text): No metrics returned
No data for video_id Do0kQEadPpA in image table.
Skipping Do0kQEadPpA (image): No metrics returned
No data for video_id -Nwa2U-7sQA in text table.
Skipping -Nwa2U-7sQA (text): No metrics returned
No data for video_id -Nwa2U-7sQA in image table.
Skipping -Nwa2U-7sQA (image): No metrics returned
🏃 View run reality show_aggregate at: https://dagshub.com/Prahaladha-Reddy/YT_RAG.mlflow/#/experiments/1/runs/7839ead0f01344deb7abb84b1443a2b5
🧪 View experiment at: https://dagshub.com/Prahaladha-Reddy/YT_RAG.mlflow/#/experiments/1
Evaluation for text modality on video_id QL2tNKHB6ew:
Total Queries: 34
Mean Rank: 17.50
MRR: 0.1211
Average Similarity: 0.7562
Hit Rate @ 1: 2.94%
Hit Rate @ 3: 8.82%
Hit Rate @ 5

In [25]:
import mlflow
import dagshub
import plotly.express as px
import pandas as pd
from mlflow.tracking import MlflowClient

# Initialize DagsHub and MLflow
client = MlflowClient()

# Fetch all experiments
experiments = client.search_experiments()
all_metrics_data = []

for experiment in experiments:
    runs = client.search_runs(experiment_ids=[experiment.experiment_id])
    for run in runs:
        for key, value in run.data.metrics.items():
            # Determine modality and clean metric name
            experiment_name = experiment.name
            if experiment_name == 'test_retrieval_version0':
                modality = 'Text' if 'text' in key else 'Image' if 'image' in key else 'Aggregate'
                metric = key.replace('text_', '').replace('image_', '').replace('avg_', '').replace('_', '')
                all_metrics_data.append({
                    'Experiment': experiment_name,
                    'Modality': modality,
                    'Metric': metric,
                    'Value': value
                })

# Convert to DataFrame
df = pd.DataFrame(all_metrics_data)

# Filter and prepare data for plotting
hit_rate_metrics = ['hitrate1', 'hitrate3', 'hitrate5']  # Adjusted for your k_values
mrr_sim_metrics = ['mrr', 'similarity']
rank_metrics = ['meanrank']

# Plot 1: Hit Rates Comparison Across Experiments
hit_df = df[df['Metric'].isin(hit_rate_metrics)]
fig1 = px.bar(hit_df, x='Metric', y='Value', color='Modality', facet_col='Experiment',
              title='Hit Rates Across Experiments',
              labels={'Value': 'Hit Rate', 'Metric': 'Rank'},
              barmode='group')
fig1.update_layout(height=600, width=1200)

# Plot 2: MRR and Similarity Comparison
mrr_sim_df = df[df['Metric'].isin(mrr_sim_metrics)]
fig2 = px.bar(mrr_sim_df, x='Metric', y='Value', color='Modality', facet_col='Experiment',
              title='MRR & Similarity Across Experiments',
              labels={'Value': 'Value', 'Metric': 'Metric'},
              barmode='group')
fig2.update_layout(height=600, width=1200)

# Plot 3: Mean Rank Comparison
rank_df = df[df['Metric'].isin(rank_metrics)]
fig3 = px.bar(rank_df, x='Experiment', y='Value', color='Modality',
              title='Mean Rank Across Experiments (Lower is Better)',
              labels={'Value': 'Mean Rank', 'Experiment': 'Experiment'},
              barmode='group')
fig3.update_layout(height=400, width=800)

# Display plots (in a Plotly-compatible environment)
fig1.show()
fig2.show()
fig3.show()

In [19]:
import pandas as pd

data = {
    "category": ["cooking", "tutorials", "Comedy", "Podcast", "News", "Coding"],
    "image_avg_hit_rate1": [2.444, 1.049, 3.175, 2.626, 1.293, 0.875],
    "image_avg_hit_rate3": [7.331, 3.146, 9.524, 7.879, 3.878, 2.625],
    "image_avg_hit_rate5": [12.22, 5.244, 15.87, 13.13, 6.464, 4.374],
    "image_avg_mean_rank": [26.83, 74.83, 21.5, 32.5, 39.75, 65.83],
    "image_avg_mrr": [0.102, 0.052, 0.124, 0.104, 0.064, 0.046],
    "image_avg_similarity": [0.331, 0.306, 0.306, 0.295, 0.311, 0.283],
    "text_avg_hit_rate1": [4.497, 1.734, 5.637, 4.887, 2.25, 1.261],
    "text_avg_hit_rate3": [13.49, 5.201, 16.91, 14.66, 6.749, 3.783],
    "text_avg_hit_rate5": [22.48, 8.669, 28.19, 24.44, 11.25, 6.304],
    "text_avg_mean_rank": [14.17, 70.67, 12, 23.83, 22.75, 61.67],
    "text_avg_mrr": [0.161, 0.076, 0.19, 0.159, 0.099, 0.06],
    "text_avg_similarity": [0.709, 0.775, 0.772, 0.794, 0.777, 0.782]
}

df = pd.DataFrame(data)
print(df)


    category  image_avg_hit_rate1  image_avg_hit_rate3  image_avg_hit_rate5  \
0    cooking                2.444                7.331               12.220   
1  tutorials                1.049                3.146                5.244   
2     Comedy                3.175                9.524               15.870   
3    Podcast                2.626                7.879               13.130   
4       News                1.293                3.878                6.464   
5     Coding                0.875                2.625                4.374   

   image_avg_mean_rank  image_avg_mrr  image_avg_similarity  \
0                26.83          0.102                 0.331   
1                74.83          0.052                 0.306   
2                21.50          0.124                 0.306   
3                32.50          0.104                 0.295   
4                39.75          0.064                 0.311   
5                65.83          0.046                 0.283   

   t

In [27]:
# pip install plotly pandas
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ------------------------------
# 1) Data
# ------------------------------
df = pd.DataFrame({
    "category": ["cooking", "tutorials", "Comedy", "Podcast", "News", "Coding"],
    "image_avg_hit_rate1": [2.444, 1.049, 3.175, 2.626, 1.293, 0.875],
    "image_avg_hit_rate3": [7.331, 3.146, 9.524, 7.879, 3.878, 2.625],
    "image_avg_hit_rate5": [12.22, 5.244, 15.87, 13.13, 6.464, 4.374],
    "image_avg_mean_rank": [26.83, 74.83, 21.5, 32.5, 39.75, 65.83],
    "image_avg_mrr": [0.102, 0.052, 0.124, 0.104, 0.064, 0.046],
    "image_avg_similarity": [0.331, 0.306, 0.306, 0.295, 0.311, 0.283],
    "text_avg_hit_rate1": [4.497, 1.734, 5.637, 4.887, 2.25, 1.261],
    "text_avg_hit_rate3": [13.49, 5.201, 16.91, 14.66, 6.749, 3.783],
    "text_avg_hit_rate5": [22.48, 8.669, 28.19, 24.44, 11.25, 6.304],
    "text_avg_mean_rank": [14.17, 70.67, 12.0, 23.83, 22.75, 61.67],
    "text_avg_mrr": [0.161, 0.076, 0.19, 0.159, 0.099, 0.06],
    "text_avg_similarity": [0.709, 0.775, 0.772, 0.794, 0.777, 0.782]
})

# Optional: consistent dark theme
px.defaults.template = "plotly_dark"
px.defaults.width = 1000
px.defaults.height = 520

# ------------------------------
# Helpers / tidy transforms
# ------------------------------
img_cols_hr = ["image_avg_hit_rate1", "image_avg_hit_rate3", "image_avg_hit_rate5"]
txt_cols_hr = ["text_avg_hit_rate1", "text_avg_hit_rate3", "text_avg_hit_rate5"]

def zscore_cols(frame, cols, invert=None):
    """z-score normalize; if a column in 'invert', flip sign so 'higher is better'."""
    X = frame[cols].copy()
    if invert:
        for c in invert:
            X[c] = -X[c]  # lower rank → higher is better
    return (X - X.mean())/X.std(ddof=0)

# ------------------------------
# 2) Grouped bars: Image hit@k
# ------------------------------
df_img = df.melt(id_vars="category", value_vars=img_cols_hr,
                 var_name="metric", value_name="value")
df_img["k"] = df_img["metric"].str.extract(r'(\d+)$')
fig1 = px.bar(df_img, x="category", y="value", color="k",
              barmode="group", title="Image: Hit Rate @k by Category",
              labels={"value":"hit rate (%)"})
fig1.update_layout(legend_title_text="@k")
fig1.show()

# ------------------------------
# 3) Grouped bars: Text hit@k
# ------------------------------
df_txt = df.melt(id_vars="category", value_vars=txt_cols_hr,
                 var_name="metric", value_name="value")
df_txt["k"] = df_txt["metric"].str.extract(r'(\d+)$')
fig2 = px.bar(df_txt, x="category", y="value", color="k",
              barmode="group", title="Text: Hit Rate @k by Category",
              labels={"value":"hit rate (%)"})
fig2.update_layout(legend_title_text="@k")
fig2.show()

# ------------------------------
# 4) Heatmap: All metrics (z-scored)
#    Invert mean_rank so brighter = better everywhere.
# ------------------------------
all_cols = [c for c in df.columns if c != "category"]
invert_cols = ["image_avg_mean_rank", "text_avg_mean_rank"]
Z = zscore_cols(df, all_cols, invert=invert_cols)
Z["category"] = df["category"]
Z = Z.set_index("category")
fig3 = px.imshow(Z.T, aspect="auto", title="Performance Heatmap (z-score, higher=better)",
                 labels=dict(x="category", y="metric", color="z"))
fig3.show()

# ------------------------------
# 5) Radar (polar): Compare categories on key metrics
#    We use normalized values; mean_rank inverted already in Z above.
# ------------------------------
radar_metrics = ["image_avg_hit_rate5","text_avg_hit_rate5",
                 "image_avg_mrr","text_avg_mrr",
                 "image_avg_similarity","text_avg_similarity",
                 "image_avg_mean_rank","text_avg_mean_rank"]
ZR = Z[radar_metrics].reset_index().rename(columns={"index":"category"})

fig4 = go.Figure()
for _, row in ZR.iterrows():
    fig4.add_trace(go.Scatterpolar(
        r=row[radar_metrics].values,
        theta=radar_metrics,
        fill="toself",
        name=row["category"]
    ))
fig4.update_layout(
    title="Radar: Normalized Key Metrics (higher=better)",
    polar=dict(radialaxis=dict(visible=True, showgrid=True))
)
fig4.show()

# ------------------------------
# 6) Scatter: Similarity vs MRR (image vs text), color by category
# ------------------------------
# ------------------------------
# 6) Scatter: Similarity vs MRR (image vs text), color by category  ✅ FIXED
# ------------------------------
sim_img = df.assign(
    modality="image",
    similarity=df["image_avg_similarity"],
    mrr=df["image_avg_mrr"],
)[["category", "modality", "similarity", "mrr"]]

sim_txt = df.assign(
    modality="text",
    similarity=df["text_avg_similarity"],
    mrr=df["text_avg_mrr"],
)[["category", "modality", "similarity", "mrr"]]

sim_mrr = pd.concat([sim_img, sim_txt], ignore_index=True)

fig5 = px.scatter(
    sim_mrr, x="similarity", y="mrr",
    color="category", symbol="modality", size="mrr",
    title="Similarity vs MRR by Modality", hover_data=["modality"]
)
fig5.update_traces(marker=dict(line=dict(width=1)))
fig5.show()

# ------------------------------
# 7) Parallel Coordinates: trade-offs across modalities
#    Normalize to [0,1]; invert ranks so higher is better.
# ------------------------------
labels_map = {
    "image_avg_hit_rate5": "Img<br>Hit@5",
    "text_avg_hit_rate5": "Txt<br>Hit@5",
    "image_avg_mrr": "Img<br>MRR",
    "text_avg_mrr": "Txt<br>MRR",
    "image_avg_similarity": "Img<br>Sim",
    "text_avg_similarity": "Txt<br>Sim",
    "image_avg_mean_rank": "Img<br>Mean<br>Rank",
    "text_avg_mean_rank": "Txt<br>Mean<br>Rank",
}

fig6 = px.parallel_coordinates(
    pc,
    color="image_avg_hit_rate5",
    dimensions=pc_cols,
    title="Parallel Coordinates: Normalized Trade-offs (higher=better)",
    labels=labels_map
)

# make it breathe
fig6.update_layout(
    width=1400,   # go larger if you have screen space
    height=650,
    margin=dict(l=110, r=110, t=80, b=80),
    font=dict(size=14)
)

# improve label/tick readability
fig6.update_traces(
    labelfont=dict(size=14),
    tickfont=dict(size=12)
)

fig6.show()  # or: fig6.show(renderer="browser") for a full-window view

# ------------------------------
# 8) Mean-rank lines (lower is better): image vs text
# ------------------------------
rank_long = df.melt(id_vars="category",
                    value_vars=["image_avg_mean_rank","text_avg_mean_rank"],
                    var_name="metric", value_name="mean_rank")
rank_long["modality"] = rank_long["metric"].str.split("_").str[0]
fig7 = px.line(rank_long, x="category", y="mean_rank", color="modality",
               markers=True, title="Mean Rank")
fig7.update_yaxes(autorange="reversed")  # emphasize lower is better
fig7.show()

# ------------------------------
# 9) Treemap: Contribution to Hit@5 by category & modality
# ------------------------------
treemap = pd.concat([
    df.assign(modality="image", hit5=df["image_avg_hit_rate5"])[["modality","category","hit5"]],
    df.assign(modality="text",  hit5=df["text_avg_hit_rate5"])[["modality","category","hit5"]],
], ignore_index=True)

fig8 = px.treemap(
    treemap,
    path=["modality","category"],
    values="hit5",
    title="Hit@5 Contribution by Category & Modality"
)
fig8.show()

# ------------------------------
# 10) Composite score leaderboard (bar)
#     Average z-scores of: hit@5 (img+text), similarity (img+text), MRR (img+text), inverse-rank (img+text).
# ------------------------------
comp_cols = ["image_avg_hit_rate5","text_avg_hit_rate5",
             "image_avg_similarity","text_avg_similarity",
             "image_avg_mrr","text_avg_mrr",
             "image_avg_mean_rank","text_avg_mean_rank"]
compZ = Z[comp_cols].mean(axis=1).rename("composite_score").reset_index()
fig9 = px.bar(compZ.sort_values("composite_score", ascending=False),
              x="category", y="composite_score",
              title="Composite Performance (avg z-score; higher=better)")
fig9.show()
